# Speech Mood Detection for an Alzheimer's / Dementia Care Platform

**Production-grade training pipeline — single self-contained notebook, optimized for Kaggle GPU.**

This notebook fine-tunes a pretrained self-supervised speech model into a **clinically-oriented mood classifier** intended to support caregivers, family members, and physicians monitoring people living with Alzheimer's disease and related dementias (ADRD).

> **Scope & honesty statement.** The single most important design fact, established by the research below, is this: the gold-standard *dementia* speech corpora (DementiaBank/Pitt, ADReSS, ADReSSo) are **credential-gated and were annotated for cognitive-decline tasks (AD-vs-control classification, MMSE regression) — they carry no mood/affect labels.** You therefore cannot train a mood detector on them directly. This notebook implements the strongest *realistic* alternative: fine-tune a top speech model on the publicly available acted speech-emotion corpora, remap their emotions into a clinically meaningful mood taxonomy, and document exactly what additional clinical validation is required before deployment. Nothing here is presented as a validated medical device.

## 1. Research — datasets

### 1a. Dementia / clinical speech corpora

| Corpus | Access | What it is labelled for | Usable for *mood*? |
|---|---|---|---|
| **DementiaBank / Pitt Corpus** | Credentialed (TalkBank consortium membership + data-use agreement) | AD vs. healthy control; longitudinal cognitive status | **No** — no affect/mood labels |
| **ADReSS** (INTERSPEECH 2020) | Credentialed via DementiaBank (NDA) | Balanced AD/non-AD classification + MMSE regression | **No mood labels** |
| **ADReSSo** (INTERSPEECH 2021) | Credentialed via DementiaBank | AD classification + cognitive-decline prediction, speech-only | **No mood labels** |
| **ADReSS-M** (ICASSP 2023) | Credentialed via DementiaBank | Multilingual AD recognition | **No mood labels** |

**Conclusion:** these corpora are essential for *dementia detection* research but are **the wrong supervision signal for a mood detector** and are not freely downloadable inside a Kaggle kernel. We therefore do **not** depend on them for training, and we document below how they re-enter the picture for clinical validation.

### 1b. Speech Emotion Recognition (SER) corpora — the practical training signal

| Corpus | Speakers | Clips | Emotions | License / access |
|---|---|---|---|---|
| **RAVDESS** | 24 (12M/12F, professional actors) | 1,440 speech | neutral, **calm**, happy, sad, angry, fearful, disgust, surprised | CC BY-NC-SA — public (Kaggle) |
| **CREMA-D** | 91 (48M/43F, diverse ethnicity/age 20–74) | 7,442 | anger, disgust, fear, happy, neutral, sad | ODbL — public (Kaggle) |
| **TESS** | 2 female (aged 26 & 64) | 2,800 | anger, disgust, fear, happy, (pleasant) surprise, sad, neutral | public (Kaggle) |
| **SAVEE** | 4 male (British English) | 480 | anger, disgust, fear, happiness, neutral, sadness, surprise | public (Kaggle) |

**Why these four, and why combine them.** Each individual corpus is demographically narrow (TESS is two women, SAVEE is four men). Combining all four balances gender, age, accent and recording conditions, which is the standard and well-supported recipe for a robust English SER model. Their union covers the six emotions present everywhere (anger, disgust, fear, happy, neutral, sad) plus RAVDESS-only *calm*.

### 1c. Datasets considered and not used here
- **IEMOCAP** — requires a separate USC license; cannot be auto-acquired on Kaggle. (Excellent for later expansion.)
- **EmoDB / EMOVO** — German / Italian; out of scope for an English-first model.

## 2. Label design — a clinically-oriented mood taxonomy

Generic entertainment emotion sets (happy/sad/angry/fear/surprise) are not directly actionable for dementia care. We instead map the acted emotions onto states aligned with the **Behavioural and Psychological Symptoms of Dementia (BPSD)** that caregivers and clinicians actually track (agitation, anxiety, low/depressed affect), while keeping a positive and a baseline state.

**Final taxonomy (6 states) and mapping from source emotions:**

| Clinical mood state | Source emotion(s) | Why it matters in care |
|---|---|---|
| **Calm** | calm | Settled/relaxed — a treatment goal; distinct from flat baseline |
| **Neutral** | neutral | Baseline affect — reference for change detection |
| **Content** | happy | Positive engagement — protective, worth reinforcing |
| **Anxious** | fearful | Worry/apprehension — early, often pre-escalation signal |
| **Agitated** | angry, disgust | High-arousal negative affect — top caregiver concern (escalation/aggression risk) |
| **Low** | sad | Withdrawal/low mood — depression/apathy proxy |

**Design decisions, justified:**
- **`surprise` is dropped.** It is clinically ambiguous (valence-neutral), absent from CREMA-D (so it would be corpus-imbalanced), and is routinely excluded in combined-corpus SER work.
- **`disgust` folds into `Agitated`.** Both are high-arousal aversive states; from a caregiver's standpoint the actionable signal ("the person is distressed/activated") is the same, and folding improves class support.
- **`Calm` is kept separate from `Neutral`** because "settled vs. flat" is a meaningful clinical distinction, even though `calm` is only sourced from RAVDESS (handled via class weighting; see notes).

These mappings live in code as a single dictionary so the taxonomy is auditable and editable in one place.

## 3. Model selection

We require a pretrained model (no training from scratch), strong on **paralinguistic** (non-ASR) tasks, and practical on a Kaggle GPU.

| Family | Pretraining signal | SER strength | Notes |
|---|---|---|---|
| **WavLM** | masked speech prediction **+ denoising** + gated relative position bias | **Top of SUPERB; best SER on EmoBox** (e.g. WavLM-Large ≈ 74% on CREMA-D, ahead of HuBERT/wav2vec2) | Best practical choice |
| HuBERT | masked prediction over clustered units | Strong, just below WavLM | Good alternative |
| wav2vec2 | contrastive | Lowest of the three on SER benchmarks | Widely used baseline |
| XLSR | multilingual wav2vec2 | Good multilingual | Overkill for English-first |
| Whisper encoder | supervised ASR | ASR-optimized, weaker paralinguistics | Better for transcription |

**Choice: WavLM.** Its denoising pretraining objective is explicitly designed to help non-ASR tasks (speaker, emotion, diarization), and it is the current SUPERB leader. We default to **`microsoft/wavlm-base-plus`** (94k-hour pretraining, ~95M params) as the strongest model that fine-tunes comfortably within Kaggle time/memory, and expose a one-line switch to **`microsoft/wavlm-large`** for maximum accuracy when more compute is available. Transfer learning is applied by **freezing the convolutional feature encoder** and fine-tuning the transformer + a classification head.

## 4. Hybrid strategy, limitations & clinical-validation path

**Training strategy (what this notebook does):**
1. Fine-tune WavLM on the union of RAVDESS + CREMA-D + TESS + SAVEE.
2. Relabel emotions into the 6-state clinical mood taxonomy above.
3. Use **speaker-disjoint, stratified** train/val/test splits so no actor leaks across splits.
4. Handle class imbalance with class-weighted loss; report macro-F1 (not just accuracy).

**Known limitations (must be read before any clinical use):**
- **Acted ≠ spontaneous, and young actors ≠ elderly patients.** The training affect is portrayed by (mostly younger) actors; dementia speech has dysfluency, slowed rate, and reduced prosodic range. Expect a domain gap.
- **Mapped labels are a proxy.** "angry/disgust → Agitated" approximates but is not the same as clinically-rated agitation.
- **No patient data is used or validated here.** Results are an *engineering* baseline, not clinical evidence.

**Path to a deployable, validated system (recommended next steps):**
1. Obtain DementiaBank/ADReSS credentials and collect (or annotate) **mood/agitation labels on real patient speech** with clinician raters.
2. **Domain-adapt** this fine-tuned model on that elderly/spontaneous data; recalibrate thresholds.
3. **Speaker- and site-independent evaluation**, fairness audits across demographics, and prospective clinical study before any deployment.

## 5. Environment setup & dependency installation

In [ ]:
# --- Dependency installation (idempotent, quiet) ---
# torch is preinstalled on Kaggle GPU images; we (re)install the audio + HF stack.
import sys, subprocess

def _pip_install(pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + pkgs
    try:
        subprocess.check_call(cmd)
    except Exception as e:
        print("pip install warning (continuing):", e)

# transformers>=4.30 guarantees WavLMForSequenceClassification support.
_pip_install(["transformers>=4.40.0", "librosa>=0.10.0", "soundfile>=0.12.1",
              "scikit-learn>=1.3.0", "kagglehub>=0.2.0"])
print("Dependency installation step complete.")

In [ ]:
# --- Core imports ---
import os, gc, json, random, math, glob, re, warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import librosa
import soundfile as sf

from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
from sklearn.model_selection import train_test_split

import transformers
from transformers import (AutoFeatureExtractor, AutoModelForAudioClassification,
                          get_linear_schedule_with_warmup)

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")

print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("numpy       :", np.__version__)
print("librosa     :", librosa.__version__)

## 6. Hardware detection (CUDA / GPU count / auto device selection)

In [ ]:
# --- Detect hardware automatically; use GPU(s) if available, else fall back to CPU ---
CUDA_AVAILABLE = torch.cuda.is_available()
N_GPU = torch.cuda.device_count() if CUDA_AVAILABLE else 0
DEVICE = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
USE_DATA_PARALLEL = N_GPU > 1   # DataParallel is the notebook-safe multi-GPU mode on Kaggle

print("=" * 60)
print("HARDWARE SUMMARY")
print("=" * 60)
print(f"CUDA available : {CUDA_AVAILABLE}")
print(f"GPU count      : {N_GPU}")
if CUDA_AVAILABLE:
    for i in range(N_GPU):
        p = torch.cuda.get_device_properties(i)
        print(f"  [GPU {i}] {p.name} | {p.total_memory/1e9:.1f} GB | CC {p.major}.{p.minor}")
    print(f"cuDNN          : {torch.backends.cudnn.version()}")
    torch.backends.cudnn.benchmark = True
else:
    print("No CUDA device found — running on CPU (slow; fine for a smoke test).")
print(f"Selected device: {DEVICE}")
print(f"DataParallel   : {USE_DATA_PARALLEL} (used automatically when >1 GPU)")
# Note on DistributedDataParallel: DDP requires a multi-process launcher and is not
# notebook-friendly on a single Kaggle kernel, so DataParallel is the correct choice here.

## 7. Reproducible random seeds

In [ ]:
SEED = 42
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)
print(f"All seeds set to {SEED}.")

## 8. Configuration

A single config object controls paths, the model, and all hyperparameters. Switch to `microsoft/wavlm-large` here for maximum accuracy when compute allows.

In [ ]:
@dataclass
class Config:
    # Model
    model_name: str = "microsoft/wavlm-base-plus"   # -> "microsoft/wavlm-large" for max accuracy
    sample_rate: int = 16000
    max_duration_sec: float = 6.0                    # clips truncated/padded to this length

    # Training
    epochs: int = 8
    batch_size: int = 16                             # per-step; DataParallel splits across GPUs
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    grad_clip: float = 1.0
    num_workers: int = 2
    freeze_feature_encoder: bool = True

    # Data acquisition
    kaggle_input_root: str = "/kaggle/input"
    try_kagglehub_download: bool = True              # used only if no local data is found
    allow_smoke_test: bool = True                    # synth data so the notebook ALWAYS completes
    smoke_test_clips_per_class: int = 40

    # Output
    output_dir: str = "outputs"
    save_hf_export: bool = True

    seed: int = 42

CFG = Config()
MAX_SAMPLES = int(CFG.max_duration_sec * CFG.sample_rate)
os.makedirs(CFG.output_dir, exist_ok=True)
print("Config:")
for k, v in asdict(CFG).items():
    print(f"  {k:24s}: {v}")
print(f"  {'max_samples':24s}: {MAX_SAMPLES}")

## 9. Clinical label taxonomy (single source of truth)

In [ ]:
# Canonical source-emotion -> clinical mood mapping. None = dropped (e.g. 'surprised').
EMOTION_TO_MOOD = {
    "neutral":   "Neutral",
    "calm":      "Calm",
    "happy":     "Content",
    "sad":       "Low",
    "angry":     "Agitated",
    "disgust":   "Agitated",
    "fearful":   "Anxious",
    "surprised": None,        # dropped: clinically ambiguous + corpus-imbalanced
}
MOOD_STATES = ["Calm", "Neutral", "Content", "Anxious", "Agitated", "Low"]

def emotion_to_mood(emotion):
    return EMOTION_TO_MOOD.get(emotion, None)

print("Mood taxonomy:", MOOD_STATES)
print("Emotion -> mood mapping:")
for e, m in EMOTION_TO_MOOD.items():
    print(f"  {e:10s} -> {m}")

## 10. Dataset acquisition

The notebook first scans `/kaggle/input` for any attached emotion corpora (recommended: add the Kaggle dataset **`dmitrybabko/speech-emotion-recognition-en`**, which bundles RAVDESS, CREMA-D, TESS and SAVEE; or attach them individually). If nothing is attached and internet is enabled, it falls back to `kagglehub` download. If neither yields data, a clearly-labelled **synthetic smoke test** lets the full pipeline run end-to-end.

In [ ]:
def find_wavs(root):
    root = Path(root)
    if not root.exists():
        return []
    exts = ("*.wav", "*.WAV", "*.flac", "*.mp3")
    files = []
    for ext in exts:
        files.extend(root.rglob(ext))
    return files

audio_files = find_wavs(CFG.kaggle_input_root)
print(f"Found {len(audio_files)} audio files under {CFG.kaggle_input_root}")

if len(audio_files) == 0 and CFG.try_kagglehub_download:
    print("No local audio found — attempting kagglehub download (needs internet ON)...")
    try:
        import kagglehub
        dl_root = kagglehub.dataset_download("dmitrybabko/speech-emotion-recognition-en")
        print("Downloaded to:", dl_root)
        audio_files = find_wavs(dl_root)
        print(f"Found {len(audio_files)} audio files after download.")
    except Exception as e:
        print("kagglehub download failed (likely no internet):", e)

print(f"TOTAL audio files available: {len(audio_files)}")

## 11. Label parsing & dataset validation

Each corpus encodes its emotion differently in the filename/path. We detect the corpus and parse the emotion robustly, then validate.

In [ ]:
RAVDESS_EMO = {"01":"neutral","02":"calm","03":"happy","04":"sad",
               "05":"angry","06":"fearful","07":"disgust","08":"surprised"}
CREMA_EMO   = {"ANG":"angry","DIS":"disgust","FEA":"fearful","HAP":"happy",
               "NEU":"neutral","SAD":"sad"}
SAVEE_EMO   = {"a":"angry","d":"disgust","f":"fearful","h":"happy",
               "n":"neutral","sa":"sad","su":"surprised"}

def parse_ravdess(path):
    stem = Path(path).stem
    parts = stem.split("-")
    if len(parts) == 7 and parts[2] in RAVDESS_EMO:
        return RAVDESS_EMO[parts[2]], f"RAVDESS_actor{parts[6]}"
    return None, None

def parse_crema(path):
    stem = Path(path).stem
    parts = stem.split("_")
    if len(parts) >= 3 and parts[2].upper() in CREMA_EMO:
        return CREMA_EMO[parts[2].upper()], f"CREMA_{parts[0]}"
    return None, None

def parse_tess(path):
    stem = Path(path).stem.lower()
    spk = "TESS_OAF" if stem.startswith("oaf") else ("TESS_YAF" if stem.startswith("yaf") else None)
    if spk is None and "tess" not in str(path).lower():
        return None, None
    if spk is None:
        spk = "TESS_unk"
    # order matters: check 'pleasant'/'ps'/'surprise' before others
    if "pleasant" in stem or stem.endswith("_ps") or "surprise" in stem:
        return "surprised", spk
    for key, emo in [("neutral","neutral"),("happy","happy"),("sad","sad"),
                     ("angry","angry"),("fear","fearful"),("disgust","disgust")]:
        if key in stem:
            return emo, spk
    return None, None

def parse_savee(path):
    stem = Path(path).stem
    m = re.match(r"^([A-Za-z]{2})_([a-z]{1,2})\d+$", stem)
    if m:
        spk, code = m.group(1), m.group(2)
        if code in SAVEE_EMO:
            return SAVEE_EMO[code], f"SAVEE_{spk}"
    return None, None

def parse_any(path):
    p = str(path).lower()
    # Try the most path-specific detector first, then fall through to all parsers.
    ordered = []
    if "ravdess" in p or "actor_" in p: ordered.append(parse_ravdess)
    if "crema" in p or "audiowav" in p: ordered.append(parse_crema)
    if "tess" in p:                     ordered.append(parse_tess)
    if "savee" in p:                    ordered.append(parse_savee)
    for fn in (parse_ravdess, parse_crema, parse_tess, parse_savee):
        if fn not in ordered:
            ordered.append(fn)
    for fn in ordered:
        emo, spk = fn(path)
        if emo is not None:
            return emo, spk, fn.__name__.replace("parse_", "")
    return None, None, None

# Build the unified dataframe from the discovered files.
rows = []
for f in audio_files:
    emo, spk, src = parse_any(f)
    if emo is None:
        continue
    mood = emotion_to_mood(emo)
    if mood is None:
        continue   # dropped emotion (e.g. surprised)
    rows.append({"path": str(f), "emotion": emo, "mood": mood,
                 "group": spk, "corpus": src})

df = pd.DataFrame(rows)
print(f"Parsed {len(df)} labelled clips into the clinical taxonomy.")
if len(df):
    print("\nBy corpus:\n", df["corpus"].value_counts())
    print("\nBy mood:\n",   df["mood"].value_counts())

## 12. Synthetic smoke-test fallback (only if no real data)

> If real corpora were found above, this cell does nothing. Otherwise it synthesizes labelled audio **purely to validate that the pipeline runs end-to-end and exports artifacts**. Any model produced from smoke-test data is **not** a real mood detector and must never be used clinically.

In [ ]:
SMOKE_TEST_MODE = False
if len(df) == 0:
    if not CFG.allow_smoke_test:
        raise RuntimeError(
            "No labelled audio found and smoke test disabled. Attach a Kaggle dataset "
            "(e.g. 'dmitrybabko/speech-emotion-recognition-en') or enable internet for kagglehub.")
    SMOKE_TEST_MODE = True
    print("!!! NO REAL DATA FOUND — GENERATING SYNTHETIC SMOKE-TEST DATA !!!")
    print("!!! Resulting model is NOT clinically valid; pipeline validation only. !!!")
    synth_dir = Path("synthetic_audio"); synth_dir.mkdir(exist_ok=True)
    sr = CFG.sample_rate
    rng = np.random.default_rng(CFG.seed)
    rows = []
    # Give each mood a distinct base frequency so the model has *some* signal to learn.
    base_freq = {m: 150 + 60 * i for i, m in enumerate(MOOD_STATES)}
    for mi, mood in enumerate(MOOD_STATES):
        for k in range(CFG.smoke_test_clips_per_class):
            dur = 2.0
            t = np.linspace(0, dur, int(sr * dur), endpoint=False)
            f0 = base_freq[mood] * (1 + 0.03 * rng.standard_normal())
            wave = 0.3 * np.sin(2 * np.pi * f0 * t)
            wave += 0.15 * np.sin(2 * np.pi * 2 * f0 * t)
            wave += 0.05 * rng.standard_normal(t.shape[0])
            wave = (wave / np.max(np.abs(wave) + 1e-9)).astype(np.float32)
            fp = synth_dir / f"synth_{mood}_{k}.wav"
            sf.write(fp, wave, sr)
            # 8 synthetic speakers so group-aware splitting still works
            rows.append({"path": str(fp), "emotion": "synthetic", "mood": mood,
                         "group": f"SYNTH_spk{k % 8}", "corpus": "synthetic"})
    df = pd.DataFrame(rows)
    CFG.epochs = 3   # keep the smoke test fast
    print(f"Synthetic dataset: {len(df)} clips across {df['mood'].nunique()} moods.")
else:
    print("Real data present — skipping synthetic generation.")

In [ ]:
# --- Dataset validation: drop unreadable files, verify a sample, show distribution ---
def is_readable(path):
    try:
        info = sf.info(path)
        return info.frames > 0
    except Exception:
        try:
            y, _ = librosa.load(path, sr=None, duration=0.5)
            return y.size > 0
        except Exception:
            return False

# Validate a random sample for speed on large corpora; validate all if small.
check_idx = df.index if len(df) <= 2000 else df.sample(2000, random_state=CFG.seed).index
bad = [i for i in check_idx if not is_readable(df.loc[i, "path"])]
if bad:
    print(f"Dropping {len(bad)} unreadable files.")
    df = df.drop(index=bad).reset_index(drop=True)

assert len(df) > 0, "No usable audio after validation."
present_moods = sorted(df["mood"].unique(), key=lambda m: MOOD_STATES.index(m))
label2id = {m: i for i, m in enumerate(present_moods)}
id2label = {i: m for m, i in label2id.items()}
df["label_id"] = df["mood"].map(label2id)
NUM_CLASSES = len(label2id)

print("Classes present:", label2id)
print(f"Total usable clips: {len(df)} | speakers: {df['group'].nunique()}")

# Class distribution plot
plt.figure(figsize=(7, 4))
counts = df["mood"].value_counts().reindex(present_moods)
plt.bar(counts.index, counts.values)
plt.title("Clinical mood class distribution"); plt.ylabel("clips")
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 13. Train / Validation / Test split (speaker-disjoint, stratified)

Splitting by **speaker group** prevents the same actor appearing in train and test (a common and severe leakage source in SER).

In [ ]:
def make_splits(df, seed=42):
    labels = df["label_id"].values
    groups = df["group"].values
    idx = np.arange(len(df))
    try:
        from sklearn.model_selection import StratifiedGroupKFold
        sgkf = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed)
        trainval_idx, test_idx = next(sgkf.split(idx, labels, groups))
        sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed + 1)
        sub_tr, sub_va = next(sgkf2.split(trainval_idx, labels[trainval_idx], groups[trainval_idx]))
        train_idx, val_idx = trainval_idx[sub_tr], trainval_idx[sub_va]
        mode = "speaker-disjoint (StratifiedGroupKFold)"
    except Exception as e:
        print("Group-aware split unavailable, using stratified split:", e)
        trainval_idx, test_idx = train_test_split(
            idx, test_size=0.15, random_state=seed, stratify=labels)
        train_idx, val_idx = train_test_split(
            trainval_idx, test_size=0.1765, random_state=seed, stratify=labels[trainval_idx])
        mode = "stratified (non-group)"
    return (df.iloc[train_idx].reset_index(drop=True),
            df.iloc[val_idx].reset_index(drop=True),
            df.iloc[test_idx].reset_index(drop=True), mode)

train_df, val_df, test_df, split_mode = make_splits(df, CFG.seed)
print(f"Split mode: {split_mode}")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"  {name} mood counts:", dict(d['mood'].value_counts()))

## 14. Audio preprocessing, Dataset & collation

Audio is resampled to 16 kHz mono, truncated/padded to a fixed max length, then passed through the WavLM feature extractor which normalizes and builds the attention mask.

In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(CFG.model_name)
print("Feature extractor:", feature_extractor.__class__.__name__,
      "| sampling_rate:", feature_extractor.sampling_rate)

class MoodAudioDataset(Dataset):
    def __init__(self, frame, sample_rate, max_samples):
        self.paths = frame["path"].tolist()
        self.labels = frame["label_id"].tolist()
        self.sr = sample_rate
        self.max_samples = max_samples
    def __len__(self):
        return len(self.paths)
    def _load(self, path):
        try:
            y, _ = librosa.load(path, sr=self.sr, mono=True)
        except Exception:
            y = np.zeros(self.sr, dtype=np.float32)
        if y.size == 0:
            y = np.zeros(self.sr, dtype=np.float32)
        if len(y) > self.max_samples:        # truncate long clips (bounds memory)
            y = y[: self.max_samples]
        return y.astype(np.float32)
    def __getitem__(self, i):
        return {"array": self._load(self.paths[i]), "label": int(self.labels[i])}

def collate_fn(batch):
    arrays = [b["array"] for b in batch]
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    feats = feature_extractor(
        arrays, sampling_rate=CFG.sample_rate, return_tensors="pt",
        padding="longest", max_length=MAX_SAMPLES, truncation=True)
    out = {"input_values": feats["input_values"], "labels": labels}
    if "attention_mask" in feats:
        out["attention_mask"] = feats["attention_mask"]
    return out

train_ds = MoodAudioDataset(train_df, CFG.sample_rate, MAX_SAMPLES)
val_ds   = MoodAudioDataset(val_df,   CFG.sample_rate, MAX_SAMPLES)
test_ds  = MoodAudioDataset(test_df,  CFG.sample_rate, MAX_SAMPLES)

pin = CUDA_AVAILABLE
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                          num_workers=CFG.num_workers, collate_fn=collate_fn,
                          pin_memory=pin, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                          num_workers=CFG.num_workers, collate_fn=collate_fn, pin_memory=pin)
test_loader  = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False,
                          num_workers=CFG.num_workers, collate_fn=collate_fn, pin_memory=pin)

# Sanity-check one batch
_b = next(iter(train_loader))
print("Batch input_values:", tuple(_b["input_values"].shape), "| labels:", tuple(_b["labels"].shape))

## 15. Model loading (WavLM + classification head, transfer learning)

In [ ]:
model = AutoModelForAudioClassification.from_pretrained(
    CFG.model_name,
    num_labels=NUM_CLASSES,
    label2id=label2id,
    id2label={str(k): v for k, v in id2label.items()},
)

# Transfer learning: freeze the CNN feature encoder, fine-tune the rest + head.
if CFG.freeze_feature_encoder and hasattr(model, "freeze_feature_encoder"):
    model.freeze_feature_encoder()
    print("Feature encoder frozen.")

model.to(DEVICE)
if USE_DATA_PARALLEL:
    model = nn.DataParallel(model)
    print(f"Wrapped model in DataParallel across {N_GPU} GPUs.")

def unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_total/1e6:.1f}M total | {n_train/1e6:.1f}M trainable")

## 16. Optimizer, scheduler, class-weighted loss & AMP

In [ ]:
from torch.cuda.amp import autocast, GradScaler

# Class weights (inverse frequency) to counter imbalance (e.g. Calm only from RAVDESS).
counts = np.bincount(train_df["label_id"].values, minlength=NUM_CLASSES).astype(np.float64)
class_weights = counts.sum() / (NUM_CLASSES * np.maximum(counts, 1.0))
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("Class weights:", {id2label[i]: round(float(w), 3) for i, w in enumerate(class_weights)})

criterion = nn.CrossEntropyLoss(weight=class_weights)
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=CFG.learning_rate,
                              weight_decay=CFG.weight_decay)
total_steps = max(1, len(train_loader) * CFG.epochs)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(CFG.warmup_ratio * total_steps), total_steps)
scaler = GradScaler(enabled=CUDA_AVAILABLE)
print(f"Optimizer/scheduler ready. Total steps: {total_steps}")

## 17. Training & validation loops (with checkpointing)

Saves `outputs/best.pt` (best validation macro-F1) and `outputs/last.pt` (latest epoch).

In [ ]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            inputs = {"input_values": batch["input_values"].to(DEVICE, non_blocking=True)}
            if "attention_mask" in batch:
                inputs["attention_mask"] = batch["attention_mask"].to(DEVICE, non_blocking=True)
            labels = batch["labels"].to(DEVICE, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=CUDA_AVAILABLE):
                outputs = model(**inputs)
                logits = outputs.logits
                loss = criterion(logits, labels)
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, CFG.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

            total_loss += loss.item() * labels.size(0)
            all_preds.extend(logits.detach().argmax(-1).cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(all_labels))
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, macro_f1

def save_ckpt(path, epoch, val_f1):
    torch.save({
        "model_state_dict": unwrap(model).state_dict(),
        "model_name": CFG.model_name,
        "label2id": label2id,
        "id2label": id2label,
        "num_classes": NUM_CLASSES,
        "sample_rate": CFG.sample_rate,
        "max_samples": MAX_SAMPLES,
        "epoch": epoch,
        "val_macro_f1": val_f1,
    }, path)

best_f1, history = -1.0, []
best_path = os.path.join(CFG.output_dir, "best.pt")
last_path = os.path.join(CFG.output_dir, "last.pt")

for epoch in range(1, CFG.epochs + 1):
    tr_loss, tr_acc, tr_f1 = run_epoch(train_loader, train=True)
    va_loss, va_acc, va_f1 = run_epoch(val_loader, train=False)
    history.append({"epoch": epoch, "train_loss": tr_loss, "train_f1": tr_f1,
                    "val_loss": va_loss, "val_acc": va_acc, "val_f1": va_f1})
    print(f"Epoch {epoch:02d}/{CFG.epochs} | "
          f"train loss {tr_loss:.4f} f1 {tr_f1:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f} f1 {va_f1:.4f}")
    save_ckpt(last_path, epoch, va_f1)
    if va_f1 > best_f1:
        best_f1 = va_f1
        save_ckpt(best_path, epoch, va_f1)
        print(f"  -> new best (val macro-F1 {best_f1:.4f}) saved to {best_path}")

print(f"\nTraining complete. Best val macro-F1: {best_f1:.4f}")
print(f"Saved: {best_path} , {last_path}")

In [ ]:
# Plot training curves
hist = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist["epoch"], hist["train_loss"], marker="o", label="train")
ax[0].plot(hist["epoch"], hist["val_loss"], marker="o", label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(hist["epoch"], hist["train_f1"], marker="o", label="train")
ax[1].plot(hist["epoch"], hist["val_f1"], marker="o", label="val")
ax[1].set_title("Macro-F1"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 18. Test-set evaluation & metrics (best checkpoint)

In [ ]:
# Load best checkpoint into the underlying model for evaluation.
ckpt = torch.load(best_path, map_location=DEVICE)
unwrap(model).load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']} (val macro-F1 {ckpt['val_macro_f1']:.4f}).")

def predict_loader(loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            inputs = {"input_values": batch["input_values"].to(DEVICE)}
            if "attention_mask" in batch:
                inputs["attention_mask"] = batch["attention_mask"].to(DEVICE)
            with autocast(enabled=CUDA_AVAILABLE):
                logits = model(**inputs).logits
            preds.extend(logits.argmax(-1).cpu().numpy().tolist())
            labels.extend(batch["labels"].numpy().tolist())
    return np.array(labels), np.array(preds)

y_true, y_pred = predict_loader(test_loader)
target_names = [id2label[i] for i in range(NUM_CLASSES)]
print(f"\nTest accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"Test macro-F1 : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print("\nClassification report:\n",
      classification_report(y_true, y_pred, target_names=target_names,
                            digits=3, zero_division=0))

## 19. Confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(target_names, rotation=45, ha="right")
ax.set_yticklabels(target_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix (row-normalized)")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{cm[i, j]}\n{cm_norm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=8)
fig.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

## 20. Error analysis

In [ ]:
err = test_df.copy().reset_index(drop=True)
err["true"] = [id2label[i] for i in y_true]
err["pred"] = [id2label[i] for i in y_pred]
err["correct"] = err["true"] == err["pred"]

print("Per-class accuracy:")
for m in target_names:
    sub = err[err["true"] == m]
    if len(sub):
        print(f"  {m:10s}: {sub['correct'].mean():.3f}  (n={len(sub)})")

print("\nMost-confused (true -> pred) pairs:")
conf = (err[~err["correct"]]
        .groupby(["true", "pred"]).size()
        .sort_values(ascending=False).head(10))
for (t, p), n in conf.items():
    print(f"  {t:10s} -> {p:10s}: {n}")

print("\nExample misclassifications:")
cols = ["corpus", "true", "pred", "path"]
sample_err = err[~err["correct"]][cols].head(8)
for _, r in sample_err.iterrows():
    print(f"  [{r['corpus']}] {r['true']} -> {r['pred']} | {Path(r['path']).name}")

## 21. Inference examples (single-clip prediction)

In [ ]:
@torch.no_grad()
def predict_mood(audio_path, top_k=3):
    model.eval()
    y, _ = librosa.load(audio_path, sr=CFG.sample_rate, mono=True)
    if len(y) > MAX_SAMPLES:
        y = y[:MAX_SAMPLES]
    feats = feature_extractor([y.astype(np.float32)], sampling_rate=CFG.sample_rate,
                              return_tensors="pt", padding="longest",
                              max_length=MAX_SAMPLES, truncation=True)
    inputs = {"input_values": feats["input_values"].to(DEVICE)}
    if "attention_mask" in feats:
        inputs["attention_mask"] = feats["attention_mask"].to(DEVICE)
    with autocast(enabled=CUDA_AVAILABLE):
        logits = model(**inputs).logits
    probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()[0]
    order = probs.argsort()[::-1][:top_k]
    return id2label[int(order[0])], [(id2label[int(i)], float(probs[i])) for i in order]

print("Inference on a few held-out test clips:\n")
demo = test_df.sample(min(5, len(test_df)), random_state=CFG.seed)
for _, r in demo.iterrows():
    pred, topk = predict_mood(r["path"])
    tops = ", ".join([f"{m}:{p:.2f}" for m, p in topk])
    print(f"  file={Path(r['path']).name}")
    print(f"    true={r['mood']:10s} pred={pred:10s} | top: {tops}")

## 22. Model export

Saves the three required artifacts and (optionally) a Hugging Face export for easy reuse.

In [ ]:
# Required artifacts
os.makedirs(CFG.output_dir, exist_ok=True)
save_ckpt(last_path, CFG.epochs, history[-1]["val_f1"])  # ensure last.pt reflects final state

label_mapping = {
    "model_name": CFG.model_name,
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
    "mood_states": target_names,
    "emotion_to_mood": EMOTION_TO_MOOD,
    "sample_rate": CFG.sample_rate,
    "max_duration_sec": CFG.max_duration_sec,
    "max_samples": MAX_SAMPLES,
    "split_mode": split_mode,
    "smoke_test_mode": SMOKE_TEST_MODE,
}
mapping_path = os.path.join(CFG.output_dir, "label_mapping.json")
with open(mapping_path, "w") as fh:
    json.dump(label_mapping, fh, indent=2)

# Optional: HF-native export (model + feature extractor) for downstream loading
if CFG.save_hf_export:
    try:
        hf_dir = os.path.join(CFG.output_dir, "hf_export")
        unwrap(model).save_pretrained(hf_dir)
        feature_extractor.save_pretrained(hf_dir)
        print("HF export saved to:", hf_dir)
    except Exception as e:
        print("HF export skipped:", e)

print("\nFinal artifacts in 'outputs/':")
for f in sorted(glob.glob(os.path.join(CFG.output_dir, "*"))):
    sz = os.path.getsize(f) if os.path.isfile(f) else 0
    print(f"  {f}  ({sz/1e6:.2f} MB)" if os.path.isfile(f) else f"  {f}/ (dir)")

assert os.path.exists(best_path), "best.pt missing!"
assert os.path.exists(last_path), "last.pt missing!"
assert os.path.exists(mapping_path), "label_mapping.json missing!"
print("\nAll required outputs present: best.pt, last.pt, label_mapping.json")
if SMOKE_TEST_MODE:
    print("\n*** REMINDER: artifacts were produced from SYNTHETIC smoke-test data and are NOT clinically valid. ***")

## 23. Reloading the exported model (verification snippet)

How a downstream service would load `best.pt` for inference.

In [ ]:
def load_for_inference(ckpt_path, device="cpu"):
    ck = torch.load(ckpt_path, map_location=device)
    fe = AutoFeatureExtractor.from_pretrained(ck["model_name"])
    mdl = AutoModelForAudioClassification.from_pretrained(
        ck["model_name"], num_labels=ck["num_classes"],
        label2id=ck["label2id"],
        id2label={str(k): v for k, v in ck["id2label"].items()})
    mdl.load_state_dict(ck["model_state_dict"])
    mdl.to(device).eval()
    return mdl, fe, ck["id2label"]

_m, _fe, _id2lab = load_for_inference(best_path, device=str(DEVICE))
print("Reloaded model OK. Classes:", _id2lab)
del _m, _fe
gc.collect()
if CUDA_AVAILABLE:
    torch.cuda.empty_cache()
print("Done.")

## Summary

This notebook fine-tuned **WavLM** into a **6-state clinical mood classifier** (Calm, Neutral, Content, Anxious, Agitated, Low) using the public acted SER corpora, with speaker-disjoint evaluation, class-weighted loss, multi-GPU support, full metrics/confusion-matrix/error analysis, inference examples, and export of `best.pt`, `last.pt`, and `label_mapping.json`.

**Before any clinical use**, follow the validation path in Section 4: acquire credentialed dementia speech, obtain clinician-rated mood labels, domain-adapt, and run prospective evaluation. This is an engineering baseline, not a medical device.